# Extract
We will be loading the dataset into a local database, which will be DuckDb. The codeblock below will be creating a `raw` schema to import and load the raw csv files.

In [1]:
import duckdb
from pathlib import Path   

# 1. Connect (creates the field if it doesn't exist)
con = duckdb.connect("../data/olist.duckdb")

# 2. Create the raw schema
con.execute("CREATE SCHEMA IF NOT EXISTS raw")

# 3. Load each CSV into raw.{table_name}
raw_dir = Path("../data/raw")
csv_files = {
    "customers":            "olist_customers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "order_payments":   "olist_order_payments_dataset.csv",
    "order_reviews":       "olist_order_reviews_dataset.csv",
    "orders":                   "olist_orders_dataset.csv",
    "products":               "olist_products_dataset.csv",
    "sellers":                   "olist_sellers_dataset.csv",
    "translation":            "olist_product_category_name_translation.csv"
}

# loop through the files and load them into duckdb
for table_name, filename in csv_files.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE raw.{table_name} AS 
        SELECT * FROM read_csv('{raw_dir / filename}', header=True, auto_detect=True)
    """)

# 4. Verify
con.execute("SHOW TABLES FROM raw").fetchdf()

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,products
7,sellers
8,translation


In [2]:
for table in csv_files.keys():
    count = con.execute(f"SELECT COUNT(*) FROM raw.{table}").fetchone()[0]
    print(f"{table}: {count:,}")

customers: 99,441
geolocation: 1,000,163
order_items: 112,650
order_payments: 103,886
order_reviews: 99,224
orders: 99,441
products: 32,951
sellers: 3,095
translation: 71


# Transform
First thing we need to is to create the schema for the resulting cleaned datasets. We will be naming it the `stage` schema.

In [3]:
# 1. Create the stage schema
con.execute("CREATE SCHEMA IF NOT EXISTS stage")

## Create Tables
##### Customer

In [ ]:
# Create stage.customers
con.execute("""
    CREATE OR REPLACE TABLE stage.orders AS
    SELECT
            customer_id,
            customer_unique_id,
            TRIM(customer_city) AS customer_city,        
            TRIM(customer_state) AS customer_state
    FROM raw.customers
""")

##### Orders

In [9]:
# Create stage.orders
con.execute("""
    CREATE OR REPLACE TABLE stage.orders AS
    SELECT
            order_id,
            customer_id,
            order_status,
            CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
            CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
            CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
            CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
            CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date,
            (order_status = 'delivered'
                AND order_approved_at IS NOT NULL
                AND order_delivered_carrier_date IS NOT NULL
                AND order_delivered_customer_date IS NOT NULL) AS is_delivery_complete
    FROM raw.orders
""")

# Load